In [1]:
!pip install tensorflow scikit-learn seaborn matplotlib numpy

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 1.8/351.2 MB 8.8 MB/s eta 0:00:40
   ---------------------------------------- 3.9/351.2 MB 9.4 MB/s eta 0:00:37
    ---------------------------------------


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed:", SEED)

In [ ]:
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print(f"GPU available: {len(gpus)}")
    for gpu in gpus:
        print(" -", gpu)
else:
    print("No GPU detected. Training will use CPU.")

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# CIFAR-10 labels are shaped as (N, 1).
# Convert them to (N,).
y_train = y_train.squeeze()
y_test = y_test.squeeze()

print("Training images :", x_train.shape)
print("Training labels :", y_train.shape)
print("Testing images  :", x_test.shape)
print("Testing labels  :", y_test.shape)

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# CIFAR-10 labels are shaped as (N, 1).
# Convert them to (N,).
y_train = y_train.squeeze()
y_test = y_test.squeeze()

print("Training images :", x_train.shape)
print("Training labels :", y_train.shape)
print("Testing images  :", x_test.shape)
print("Testing labels  :", y_test.shape)

In [ ]:
print("Image dimensions:", x_train.shape[1:])
print("Number of training images:", len(x_train))
print("Number of testing images:", len(x_test))
print("Number of classes:", NUM_CLASSES)
print("Pixel value range:", x_train.min(), "to", x_train.max())

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

sample_indices = np.random.choice(len(x_train), size=10, replace=False)

for ax, idx in zip(axes.flat, sample_indices):
    ax.imshow(x_train[idx])
    ax.set_title(CLASS_NAMES[y_train[idx]])
    ax.axis("off")

plt.suptitle("CIFAR-10 Sample Images", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

sample_indices = np.random.choice(len(x_train), size=10, replace=False)

for ax, idx in zip(axes.flat, sample_indices):
    ax.imshow(x_train[idx])
    ax.set_title(CLASS_NAMES[y_train[idx]])
    ax.axis("off")

plt.suptitle("CIFAR-10 Sample Images", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
x_train_normalized = x_train.astype("float32") / 255.0
x_test_normalized = x_test.astype("float32") / 255.0

print("Original range:", x_train.min(), "to", x_train.max())
print(
    "Normalized range:",
    x_train_normalized.min(),
    "to",
    x_train_normalized.max()
)

In [ ]:
input_matrix = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
], dtype=np.float32)

kernel = np.array([
    [1, 0],
    [0, 1]
], dtype=np.float32)

output_height = input_matrix.shape[0] - kernel.shape[0] + 1
output_width = input_matrix.shape[1] - kernel.shape[1] + 1

feature_map = np.zeros((output_height, output_width))

for i in range(output_height):
    for j in range(output_width):
        window = input_matrix[
            i:i + kernel.shape[0],
            j:j + kernel.shape[1]
        ]

        feature_map[i, j] = np.sum(window * kernel)

print("Input:")
print(input_matrix)

print("\nKernel:")
print(kernel)

print("\nFeature Map:")
print(feature_map)

In [ ]:
def conv_output_size(n, f, p, s):
    """
    Calculate convolution output size for one spatial dimension.

    Parameters
    ----------
    n : int
        Input dimension.
    f : int
        Filter/kernel size.
    p : int
        Padding.
    s : int
        Stride.
    """
    return ((n - f + 2 * p) // s) + 1


def same_output_size(n, s):
    """
    Output size for TensorFlow/Keras 'same' padding.
    """
    return int(np.ceil(n / s))

In [ ]:
sample_image = x_train_normalized[0]

kernel_sizes = [3, 5, 7]

print("Input shape:", sample_image.shape)
print()

for kernel_size in kernel_sizes:
    conv_layer = layers.Conv2D(
        filters=16,
        kernel_size=(kernel_size, kernel_size),
        strides=(1, 1),
        padding="valid",
        use_bias=True
    )

    input_tensor = tf.expand_dims(sample_image, axis=0)
    output = conv_layer(input_tensor)

    print(
        f"Kernel {kernel_size}x{kernel_size} -> "
        f"Feature map shape: {output.shape}"
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, kernel_size in zip(axes, kernel_sizes):
    conv_layer = layers.Conv2D(
        filters=1,
        kernel_size=(kernel_size, kernel_size),
        padding="same",
        use_bias=False
    )

    output = conv_layer(
        tf.expand_dims(sample_image, axis=0)
    )

    feature = output[0, :, :, 0].numpy()

    ax.imshow(feature, cmap="gray")
    ax.set_title(f"{kernel_size}×{kernel_size} Kernel")
    ax.axis("off")

plt.suptitle("Feature Maps for Different Kernel Sizes")
plt.tight_layout()
plt.show()

In [ ]:
N = 32
F = 3

configurations = [
    ("Stride 1, Valid", 1, 0),
    ("Stride 2, Valid", 2, 0),
    ("Stride 1, Same", 1, None),
    ("Stride 2, Same", 2, None),
]

for name, stride, padding in configurations:

    if padding is None:
        output = same_output_size(N, stride)
    else:
        output = conv_output_size(N, F, padding, stride)

    print(f"{name:<20} -> Output size: {output} × {output}")

In [ ]:
input_tensor = tf.expand_dims(sample_image, axis=0)

experiments = [
    ("Stride 1 - Valid", 1, "valid"),
    ("Stride 2 - Valid", 2, "valid"),
    ("Stride 1 - Same", 1, "same"),
    ("Stride 2 - Same", 2, "same"),
]

for name, stride, padding in experiments:

    conv = layers.Conv2D(
        filters=16,
        kernel_size=3,
        strides=stride,
        padding=padding
    )

    output = conv(input_tensor)

    print(f"{name:<22} -> {output.shape}")

In [ ]:
kernels = {
    "Identity": np.array([
        [0, 0, 0],
        [0, 1, 0],
        [0, 0, 0]
    ], dtype=np.float32),

    "Sobel-X": np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ], dtype=np.float32),

    "Sobel-Y": np.array([
        [-1, -2, -1],
        [0, 0, 0],
        [1, 2, 1]
    ], dtype=np.float32),

    "Laplacian": np.array([
        [0, -1, 0],
        [-1, 4, -1],
        [0, -1, 0]
    ], dtype=np.float32),

    "Sharpen": np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ], dtype=np.float32),

    "Box Blur": np.ones((3, 3), dtype=np.float32) / 9,

    "Gaussian Blur": np.array([
        [1, 2, 1],
        [2, 4, 2],
        [1, 2, 1]
    ], dtype=np.float32) / 16,

    "Emboss": np.array([
        [-2, -1, 0],
        [-1, 1, 1],
        [0, 1, 2]
    ], dtype=np.float32),

    "Outline": np.array([
        [-1, -1, -1],
        [-1, 8, -1],
        [-1, -1, -1]
    ], dtype=np.float32),

    "Motion Blur": np.eye(5, dtype=np.float32) / 5
}

for name, kernel in kernels.items():
    print(f"\n{name}:")
    print(kernel)

In [ ]:
# Convert sample RGB image to grayscale
sample_gray = tf.image.rgb_to_grayscale(
    tf.expand_dims(sample_image, axis=0)
)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))

for ax, (name, kernel) in zip(axes.flat, kernels.items()):

    kernel_tensor = tf.constant(
        kernel.reshape(kernel.shape[0], kernel.shape[1], 1, 1),
        dtype=tf.float32
    )

    filtered = tf.nn.conv2d(
        sample_gray,
        kernel_tensor,
        strides=1,
        padding="SAME"
    )

    result = filtered[0, :, :, 0].numpy()

    ax.imshow(result, cmap="gray")
    ax.set_title(name)
    ax.axis("off")

plt.suptitle("Classical Convolution Kernels", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
feature_map = np.array([
    [1, 5, 2, 3],
    [7, 8, 1, 0],
    [4, 6, 9, 5],
    [2, 3, 1, 8]
], dtype=np.float32)

pool_size = 2
stride = 2

output_height = (feature_map.shape[0] - pool_size) // stride + 1
output_width = (feature_map.shape[1] - pool_size) // stride + 1

max_pool_output = np.zeros((output_height, output_width))

for i in range(output_height):
    for j in range(output_width):

        window = feature_map[
            i * stride:i * stride + pool_size,
            j * stride:j * stride + pool_size
        ]

        max_pool_output[i, j] = np.max(window)

print("Input feature map:")
print(feature_map)

print("\nMax pooling output:")
print(max_pool_output)

In [ ]:
input_pool = tf.reshape(feature_map, (1, 4, 4, 1))

max_pool = layers.MaxPooling2D(
    pool_size=2,
    strides=2
)

avg_pool = layers.AveragePooling2D(
    pool_size=2,
    strides=2
)

max_output = max_pool(input_pool).numpy()[0, :, :, 0]
avg_output = avg_pool(input_pool).numpy()[0, :, :, 0]

print("Max Pooling:")
print(max_output)

print("\nAverage Pooling:")
print(avg_output)

print("\nMax Pooling output shape:", max_output.shape)
print("Average Pooling output shape:", avg_output.shape)

In [ ]:
input_channels = 3
filters = 16
kernel_size = 3

parameters = (
    kernel_size * kernel_size * input_channels + 1
) * filters

print("Trainable parameters:", parameters)

In [ ]:
N = 64
F = 5
S = 2
P = 2

output_size = conv_output_size(N, F, P, S)

print("Output size:", output_size)
print(f"Output dimensions: {output_size} × {output_size}")

In [ ]:
filters = 64
kernel_size = 3
input_channels = 3

parameters = (
    kernel_size * kernel_size * input_channels + 1
) * filters

print("Trainable parameters:", parameters)

In [ ]:
x = np.linspace(-6, 6, 500)

relu = np.maximum(0, x)
sigmoid = 1 / (1 + np.exp(-x))

plt.figure(figsize=(10, 6))

plt.plot(x, relu, label="ReLU")
plt.plot(x, sigmoid, label="Sigmoid")

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)

plt.title("ReLU vs Sigmoid")
plt.xlabel("Input")
plt.ylabel("Activation")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
VALIDATION_SIZE = 5000

x_validation = x_train_normalized[-VALIDATION_SIZE:]
y_validation = y_train[-VALIDATION_SIZE:]

x_training = x_train_normalized[:-VALIDATION_SIZE]
y_training = y_train[:-VALIDATION_SIZE]

print("Training:", x_training.shape)
print("Validation:", x_validation.shape)
print("Testing:", x_test_normalized.shape)

In [ ]:
def build_cnn(pooling_type="max"):
    """
    Build the CNN required by the experiment.

    pooling_type:
        'max' -> MaxPooling2D
        'avg' -> AveragePooling2D
    """

    if pooling_type == "max":
        PoolingLayer = layers.MaxPooling2D
    elif pooling_type == "avg":
        PoolingLayer = layers.AveragePooling2D
    else:
        raise ValueError("pooling_type must be 'max' or 'avg'")

    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),

        layers.Conv2D(
            filters=16,
            kernel_size=3,
            padding="same"
        ),
        layers.ReLU(),

        PoolingLayer(
            pool_size=2,
            strides=2
        ),

        layers.Conv2D(
            filters=32,
            kernel_size=3,
            padding="same"
        ),
        layers.ReLU(),

        PoolingLayer(
            pool_size=2,
            strides=2
        ),

        layers.Flatten(),

        layers.Dense(128),
        layers.ReLU(),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ])

    return model

In [ ]:
model = build_cnn(pooling_type="max")

model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

In [ ]:
EPOCHS = 20
BATCH_SIZE = 32

history = model.fit(
    x_training,
    y_training,
    validation_data=(x_validation, y_validation),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

In [ ]:
history_dict = history.history

print("Available metrics:")
for key in history_dict:
    print("-", key)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(
    x_test_normalized,
    y_test,
    batch_size=BATCH_SIZE,
    verbose=1
)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")

In [ ]:
probabilities = model.predict(
    x_test_normalized,
    batch_size=BATCH_SIZE,
    verbose=1
)

y_pred = np.argmax(probabilities, axis=1)

print("Prediction shape:", y_pred.shape)
print("Actual labels shape:", y_test.shape)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

In [ ]:
report = classification_report(
    y_test,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
)

print(report)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(11, 9))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)

plt.title("CIFAR-10 Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.tight_layout()
plt.show()

In [ ]:
feature_extractor = keras.Model(
    inputs=model.input,
    outputs=model.layers[0].output
)

sample = x_test_normalized[0:1]

feature_maps = feature_extractor.predict(
    sample,
    verbose=0
)

print("Feature map tensor shape:", feature_maps.shape)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0, :, :, i], cmap="viridis")
    ax.set_title(f"Feature Map {i + 1}")
    ax.axis("off")

plt.suptitle("Feature Maps from First Convolution Layer", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

axes[0, 0].imshow(x_test[0])
axes[0, 0].set_title(
    f"Original: {CLASS_NAMES[y_test[0]]}"
)
axes[0, 0].axis("off")

for i, ax in enumerate(axes.flat[1:]):
    ax.imshow(feature_maps[0, :, :, i], cmap="viridis")
    ax.set_title(f"Feature Map {i + 1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
avg_model = build_cnn(pooling_type="avg")

avg_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

avg_model.summary()

In [ ]:
avg_history = avg_model.fit(
    x_training,
    y_training,
    validation_data=(x_validation, y_validation),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

In [ ]:
avg_test_loss, avg_test_accuracy = avg_model.evaluate(
    x_test_normalized,
    y_test,
    batch_size=BATCH_SIZE,
    verbose=1
)

print(f"Average Pooling Test Loss     : {avg_test_loss:.4f}")
print(f"Average Pooling Test Accuracy : {avg_test_accuracy:.4f}")
print(f"Average Pooling Accuracy      : {avg_test_accuracy * 100:.2f}%")

In [ ]:
max_accuracy = test_accuracy
average_accuracy = avg_test_accuracy

pooling_results = {
    "Max Pooling": max_accuracy,
    "Average Pooling": average_accuracy
}

plt.figure(figsize=(8, 6))

bars = plt.bar(
    pooling_results.keys(),
    pooling_results.values()
)

plt.title("Max Pooling vs Average Pooling")
plt.ylabel("Test Accuracy")
plt.ylim(0, 1)

for bar, value in zip(bars, pooling_results.values()):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.01,
        f"{value:.4f}",
        ha="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
difference = max_accuracy - average_accuracy

print(
    f"Max Pooling Accuracy     : {max_accuracy * 100:.2f}%"
)
print(
    f"Average Pooling Accuracy : {average_accuracy * 100:.2f}%"
)
print(
    f"Difference               : {abs(difference) * 100:.2f} percentage points"
)

if difference > 0:
    print(
        "\nInference: Max pooling produced higher test accuracy in this experiment. "
        "It retains the strongest activation within each pooling region."
    )
else:
    print(
        "\nInference: Average pooling produced higher test accuracy in this experiment. "
        "It aggregates activations within each pooling region and can provide smoother representations."
    )

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["val_accuracy"],
    label="Max Pooling"
)

plt.plot(
    avg_history.history["val_accuracy"],
    label="Average Pooling"
)

plt.title("Validation Accuracy: Max vs Average Pooling")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
def build_filter_comparison_model(filters=16):
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),

        layers.Conv2D(
            filters=filters,
            kernel_size=3,
            padding="same",
            activation="relu"
        ),

        layers.MaxPooling2D(pool_size=2),

        layers.Conv2D(
            filters=filters * 2,
            kernel_size=3,
            padding="same",
            activation="relu"
        ),

        layers.MaxPooling2D(pool_size=2),

        layers.Flatten(),

        layers.Dense(128, activation="relu"),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ])

    return model

In [ ]:
model_16 = build_filter_comparison_model(filters=16)
model_64 = build_filter_comparison_model(filters=64)

print("16-filter model")
print("-" * 50)
print("Parameters:", model_16.count_params())

print("\n64-filter model")
print("-" * 50)
print("Parameters:", model_64.count_params())

In [ ]:
model_64.compile(
    optimizer=keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_64 = model_64.fit(
    x_training,
    y_training,
    validation_data=(x_validation, y_validation),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

In [ ]:
loss_64, accuracy_64 = model_64.evaluate(
    x_test_normalized,
    y_test,
    batch_size=BATCH_SIZE,
    verbose=1
)

print(f"64-filter Test Accuracy: {accuracy_64 * 100:.2f}%")
print(f"64-filter Parameters    : {model_64.count_params():,}")

In [ ]:
print("Filter Comparison")
print("=" * 60)

print(
    f"16-filter model parameters : {model_16.count_params():,}"
)

print(
    f"64-filter model parameters : {model_64.count_params():,}"
)

print(
    f"\n64-filter model accuracy   : {accuracy_64 * 100:.2f}%"
)

print(
    f"Original 16-filter accuracy: {test_accuracy * 100:.2f}%"
)

In [ ]:
filter_accuracies = [
    test_accuracy,
    accuracy_64
]

filter_labels = [
    "16 Filters",
    "64 Filters"
]

plt.figure(figsize=(8, 6))

bars = plt.bar(
    filter_labels,
    filter_accuracies
)

plt.title("Effect of Number of Filters on Test Accuracy")
plt.ylabel("Test Accuracy")
plt.ylim(0, 1)

for bar, value in zip(bars, filter_accuracies):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.01,
        f"{value:.4f}",
        ha="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
results = {
    "Training Accuracy": history.history["accuracy"][-1],
    "Validation Accuracy": history.history["val_accuracy"][-1],
    "Testing Accuracy": test_accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    "Number of Parameters": model.count_params()
}

print("=" * 60)
print("FINAL CNN RESULTS")
print("=" * 60)

for metric, value in results.items():

    if "Parameters" in metric:
        print(f"{metric:<25}: {value:,}")
    else:
        print(f"{metric:<25}: {value:.4f}")

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "Metric": [
        "Training Accuracy",
        "Validation Accuracy",
        "Testing Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Number of Parameters"
    ],
    "Value": [
        results["Training Accuracy"],
        results["Validation Accuracy"],
        results["Testing Accuracy"],
        results["Precision"],
        results["Recall"],
        results["F1-score"],
        results["Number of Parameters"]
    ]
})

results_df

In [ ]:
discussion_answers = {
    "1. Why is convolution preferred over fully connected layers for images?":
        "Convolution exploits the spatial structure of images using local receptive "
        "fields and shared weights. This drastically reduces parameters while "
        "allowing the network to detect local patterns such as edges and textures.",

    "2. How does stride affect the feature map size?":
        "Increasing the stride moves the kernel by larger steps, reducing the "
        "spatial dimensions of the resulting feature map. Therefore, larger "
        "stride generally produces smaller feature maps.",

    "3. What is the role of padding?":
        "Padding adds additional pixels around the input boundaries. It can "
        "preserve spatial dimensions, allow edge information to be processed, "
        "and control the output feature-map size.",

    "4. Why is pooling used?":
        "Pooling reduces the spatial dimensions of feature maps, decreasing "
        "computational cost and providing some translational robustness. "
        "Max pooling retains strong activations while average pooling computes "
        "the average activation within a region.",

    "5. How do feature maps represent image characteristics?":
        "A feature map represents the response of a convolutional filter to "
        "specific patterns in an image. Early layers generally detect low-level "
        "features such as edges and textures, while deeper layers can represent "
        "more complex structures.",

    "6. Why do CNNs require fewer parameters than MLPs?":
        "CNNs use local connectivity and weight sharing. The same convolution "
        "filter is applied across different spatial locations, so the network "
        "does not require separate weights for every input pixel-to-neuron connection."
    }

}

for question, answer in discussion_answers.items():
    print("\n" + question)
    print("-" * len(question))
    print(answer)

In [ ]:
comparison = pd.DataFrame({
    "Experiment": [
        "Max Pooling",
        "Average Pooling",
        "16 Filters",
        "64 Filters"
    ],
    "Test Accuracy": [
        test_accuracy,
        avg_test_accuracy,
        test_accuracy,
        accuracy_64
    ],
    "Parameters": [
        model.count_params(),
        avg_model.count_params(),
        model_16.count_params(),
        model_64.count_params()
    ]
})

comparison["Test Accuracy (%)"] = (
    comparison["Test Accuracy"] * 100
)

comparison

In [ ]:
## Conclusion

A Convolutional Neural Network was implemented using TensorFlow/Keras for
classification of CIFAR-10 images. The experiment demonstrated the effect of
convolution kernel size, stride, padding, pooling operations and number of
filters on CNN feature extraction and classification performance.

The CNN was trained using the Adam optimizer for 20 epochs with a batch size
of 32. Training/validation curves, convolutional feature maps, confusion
matrix, classification metrics and pooling/filter comparisons were analyzed.

The experiment demonstrates that CNNs are well suited for image classification
because convolution exploits spatial locality and weight sharing, while pooling
reduces spatial dimensions and computational requirements.